# IBM Applied Data Science Capstone — Falcon 9 First-Stage Landing Prediction

## Purpose
This notebook completes the predictive-analysis evidence required by the IBM capstone: train Logistic Regression, SVM, Decision Tree, and KNN classifiers; tune them with cross-validation; compare metrics; select a model using training-only cross-validation results; and generate the **actual held-out test confusion matrix**.

## Data files
Place the IBM course files in `data/`:
- `dataset_part_2.csv` — course dataset containing the binary target `Class`.
- `dataset_part_3.csv` — course one-hot encoded feature matrix (preferred). If absent, the notebook builds a preprocessing pipeline from `dataset_part_2.csv`.

Do not copy performance figures from another submission. The outputs below are produced when this notebook is run with your data.

In [ ]:
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    jaccard_score, balanced_accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from IPython.display import display

RANDOM_STATE = 2
DATA_DIR = Path("data")
TARGET = "Class"

## 1. Load and validate the IBM course data

The target must be `Class`, where 1 means a successful first-stage landing and 0 means an unsuccessful first-stage landing. This target is not overall mission success.

In [ ]:
def find_file(filename):
    candidates = [DATA_DIR / filename, Path(filename), Path("../data") / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

part2_path = find_file("dataset_part_2.csv")
part3_path = find_file("dataset_part_3.csv")

if part2_path is None:
    raise FileNotFoundError(
        "dataset_part_2.csv not found. Put the IBM course file in the project's data/ folder."
    )

raw = pd.read_csv(part2_path)
raw.columns = [str(c).strip() for c in raw.columns]
if TARGET not in raw.columns:
    raise ValueError(f"Expected target column '{TARGET}' in {part2_path}; found {list(raw.columns)}")

# Remove accidental exported index columns.
raw = raw.loc[:, ~raw.columns.astype(str).str.match(r"^Unnamed")]
y = pd.to_numeric(raw[TARGET], errors="coerce")
if y.isna().any() or not set(y.unique()).issubset({0, 1}):
    raise ValueError("Class must contain only numeric 0/1 values with no missing targets.")
y = y.astype(int)

if part3_path is not None:
    X = pd.read_csv(part3_path)
    X.columns = [str(c).strip() for c in X.columns]
    X = X.loc[:, ~X.columns.astype(str).str.match(r"^Unnamed")]
    if TARGET in X.columns:
        X = X.drop(columns=[TARGET])
    if len(X) != len(y):
        raise ValueError(
            f"Row mismatch: dataset_part_2 has {len(y)} rows but dataset_part_3 has {len(X)}. "
            "Confirm both files are from the same IBM lab run and in the same row order."
        )
    # The IBM part_3 file is expected to be one-hot encoded numeric features.
    nonnumeric = X.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
    if nonnumeric:
        raise ValueError(
            "dataset_part_3 should be numeric/one-hot encoded. Non-numeric columns: "
            + str(nonnumeric) + ". Use the raw-data fallback by temporarily moving part_3 out of data/."
        )
    X = X.apply(pd.to_numeric, errors="coerce")
    feature_source = str(part3_path)
else:
    # Fallback when the one-hot encoded part_3 file is not available.
    X = raw.drop(columns=[TARGET]).copy()
    feature_source = str(part2_path) + " (raw features; preprocessing pipeline applied)"

# Ensure no target leakage through a duplicated target-like feature.
leakage_cols = [c for c in X.columns if re.sub(r"[^a-z]", "", c.lower()) in {"class", "landingsuccess", "landingoutcome"}]
if leakage_cols:
    X = X.drop(columns=leakage_cols)

print("Feature source:", feature_source)
print("Rows:", len(X), " | Features:", X.shape[1])
print("Target distribution:")
display(y.value_counts().rename(index={0: "Unsuccessful (0)", 1: "Successful (1)"}).to_frame("Records"))
print("Missing target values:", int(y.isna().sum()))
print("Duplicate feature rows:", int(X.duplicated().sum()))

## 2. Train/test split and preprocessing

The test set is held out and is not used to tune hyperparameters. Scaling is fitted inside each training fold through a pipeline, preventing data leakage from the test set into model training. Stratification preserves the class balance as far as possible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print("Training class distribution:")
display(y_train.value_counts(normalize=True).sort_index().rename("Share").to_frame())

# All numeric features: median imputation + scaling.
# If the raw fallback is used, categorical columns are one-hot encoded within CV folds.
num_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

transformers = []
if num_cols:
    transformers.append(("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols))
if cat_cols:
    transformers.append(("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

min_class_count = int(y_train.value_counts().min())
n_splits = min(10, min_class_count)
if n_splits < 2:
    raise ValueError("Not enough examples in the minority class for stratified cross-validation.")
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print(f"Cross-validation folds: {n_splits}")

## 3. Train and tune four classification models

Grid search is performed on the training partition only. The primary selection metric is mean cross-validation accuracy; mean cross-validation F1 is retained as a transparent tie-breaker. The held-out test set is reported once after model selection.

In [ ]:
model_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        {"model__C": [0.01, 0.1, 1, 10], "model__solver": ["liblinear", "lbfgs"]}
    ),
    "Support Vector Machine": (
        SVC(probability=True, random_state=RANDOM_STATE),
        {"model__C": [0.1, 1, 10], "model__kernel": ["linear", "rbf", "sigmoid"], "model__gamma": ["scale", "auto"]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {"model__criterion": ["gini", "entropy"], "model__max_depth": [None, 3, 5, 8],
         "model__min_samples_split": [2, 5, 10], "model__min_samples_leaf": [1, 2, 4]}
    ),
    "K-Nearest Neighbors": (
        KNeighborsClassifier(),
        {"model__n_neighbors": [3, 5, 7, 9], "model__weights": ["uniform", "distance"],
         "model__p": [1, 2]}
    ),
}

searches = {}
cv_rows = []
for name, (estimator, grid) in model_grids.items():
    pipe = Pipeline([( "prep", preprocessor), ("model", estimator)])
    search = GridSearchCV(
        pipe, param_grid=grid, scoring={"accuracy": "accuracy", "f1": "f1", "jaccard": "jaccard"},
        refit="accuracy", cv=cv, n_jobs=-1, return_train_score=False, error_score="raise"
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best_idx = search.best_index_
    cv_rows.append({
        "Model": name,
        "Best CV accuracy": search.cv_results_["mean_test_accuracy"][best_idx],
        "CV accuracy std": search.cv_results_["std_test_accuracy"][best_idx],
        "CV F1": search.cv_results_["mean_test_f1"][best_idx],
        "CV Jaccard": search.cv_results_["mean_test_jaccard"][best_idx],
        "Best parameters": search.best_params_,
    })
    print(name, "completed")

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["Best CV accuracy", "CV F1"], ascending=False
).reset_index(drop=True)
display(cv_results.drop(columns=["Best parameters"]).style.format({
    "Best CV accuracy": "{:.3f}", "CV accuracy std": "{:.3f}", "CV F1": "{:.3f}", "CV Jaccard": "{:.3f}"
}))

## 4. Select the model using cross-validation results

Selection uses training-only cross-validation performance. If two models tie on mean CV accuracy, mean CV F1 is used as the tie-breaker. Test-set performance is **not** used to choose the winner.

In [ ]:
best_name = cv_results.iloc[0]["Model"]
selected_search = searches[best_name]
selected_model = selected_search.best_estimator_
print("Selected model:", best_name)
print("Selection basis: highest mean cross-validation accuracy, then CV F1.")
print("Best hyperparameters:")
print(selected_search.best_params_)

## 5. Evaluate all models on the held-out test set

The following table reports the actual test-set results generated by this notebook. Accuracy alone is not enough; Jaccard, precision, recall, F1, and balanced accuracy add information about class-specific performance.

In [ ]:
test_rows = []
test_predictions = {}
for name, search in searches.items():
    pred = search.best_estimator_.predict(X_test)
    test_predictions[name] = pred
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Jaccard": jaccard_score(y_test, pred, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_test, pred),
    })

test_results = pd.DataFrame(test_rows).sort_values(
    ["Accuracy", "F1"], ascending=False
).reset_index(drop=True)
display(test_results.style.format({c: "{:.3f}" for c in test_results.columns if c != "Model"}))
print("Selected model from training CV:", best_name)

## 6. Confusion matrix for the selected model — required submission evidence

This cell produces the actual 2×2 matrix for the selected model on the held-out test set. Save or screenshot this output and insert it into the final presentation. Do not type in values copied from an example or another student's work.

In [ ]:
y_pred_selected = selected_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_selected, labels=[0, 1])
print("Selected model:", best_name)
print("Rows = actual [0, 1]; columns = predicted [0, 1]")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Unsuccessful (0)", "Successful (1)"]
)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
Path("outputs").mkdir(exist_ok=True)
fig.savefig("outputs/selected_model_confusion_matrix.png", dpi=220, bbox_inches="tight")
plt.show()

print("Classification report:")
print(classification_report(
    y_test, y_pred_selected,
    labels=[0, 1],
    target_names=["Unsuccessful landing", "Successful landing"],
    zero_division=0,
))

## 7. Export results for the presentation

This cell creates reproducible files that can be inserted into the capstone deck. Include the model comparison table, confusion matrix image, selected model name, and the test-set size in your predictive-analysis slides.

In [ ]:
Path("outputs").mkdir(exist_ok=True)
cv_results.to_csv("outputs/model_cross_validation_results.csv", index=False)
test_results.to_csv("outputs/model_test_results.csv", index=False)
pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]).to_csv(
    "outputs/selected_model_confusion_matrix.csv"
)
summary = pd.DataFrame([{
    "Selected model": best_name,
    "Training rows": len(X_train),
    "Testing rows": len(X_test),
    "Features": X.shape[1],
    "Test accuracy": accuracy_score(y_test, y_pred_selected),
    "Test precision": precision_score(y_test, y_pred_selected, zero_division=0),
    "Test recall": recall_score(y_test, y_pred_selected, zero_division=0),
    "Test F1": f1_score(y_test, y_pred_selected, zero_division=0),
    "Test Jaccard": jaccard_score(y_test, y_pred_selected, zero_division=0),
}])
summary.to_csv("outputs/selected_model_summary.csv", index=False)
display(summary)
print("Saved evidence to outputs/.")

## 8. Interpretation guidance for the presentation

Write the conclusion from the values generated above, not from a prior example. State:
1. Which model was selected and why (CV accuracy and the CV F1 tie-breaker).
2. The selected model's held-out test accuracy, precision, recall, F1, and Jaccard score.
3. The four confusion-matrix cells (TN, FP, FN, TP).
4. The operational implication of false positives and false negatives.
5. Limitations: small historical dataset, time period, class balance, proxy features, and the fact that retrospective classification is not a validated engineering safety system.

**Reproducibility note:** the IBM capstone is a learning exercise. Report the course sample and split used. Do not generalize the score to future launches without temporal validation and independent data.